In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("Nassau_Candy_Distributor.csv")
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

In [3]:
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True, errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True, errors="coerce")

In [4]:
df["Lead Time"] = (df["Ship Date"] - df["Order Date"]).dt.days

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          10194 non-null  int64         
 1   Order ID        10194 non-null  object        
 2   Order Date      10194 non-null  datetime64[ns]
 3   Ship Date       10194 non-null  datetime64[ns]
 4   Ship Mode       10194 non-null  object        
 5   Customer ID     10194 non-null  int64         
 6   Country/Region  10194 non-null  object        
 7   City            10194 non-null  object        
 8   State/Province  10194 non-null  object        
 9   Postal Code     10194 non-null  object        
 10  Division        10194 non-null  object        
 11  Region          10194 non-null  object        
 12  Product ID      10194 non-null  object        
 13  Product Name    10194 non-null  object        
 14  Sales           10194 non-null  float64       
 15  Un

In [6]:
df.describe()

,Row ID,Order Date,Ship Date,Customer ID,Sales,Units,Gross Profit,Cost,Lead Time
count,10194.000000,10194,10194,10194.000000,10194.000000,10194.000000,10194.000000,10194.000000,10194.000000
mean,5097.500000,2025-03-13 03:08:26.415538176,2028-10-23 23:20:43.790465024,134468.961154,13.908537,3.791838,9.166451,4.742087,1320.841868
min,1.000000,2024-01-02 00:00:00,2026-06-30 00:00:00,100006.000000,1.250000,1.000000,0.250000,0.600000,904.000000
25%,2549.250000,2024-09-28 00:00:00,2027-11-09 00:00:00,117212.000000,7.200000,2.000000,4.900000,2.400000,1271.000000
50%,5097.500000,2025-04-06 12:00:00,2028-12-18 00:00:00,133550.000000,10.800000,3.000000,7.470000,3.600000,1274.000000
75%,7645.750000,2025-09-16 00:00:00,2029-11-08 00:00:00,152051.000000,18.000000,5.000000,12.250000,5.700000,1638.000000
max,10194.000000,2025-12-31 00:00:00,2030-06-28 00:00:00,192314.000000,260.000000,14.000000,130.000000,130.000000,1642.000000
std,2942.898656,NaN,NaN,20231.483007,11.341020,2.228317,6.643740,5.061647,262.444892


In [7]:
print(df["Lead Time"].describe())
print(df.isnull().sum())

count    10194.000000
mean      1320.841868
std        262.444892
min        904.000000
25%       1271.000000
50%       1274.000000
75%       1638.000000
max       1642.000000
Name: Lead Time, dtype: float64
Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Division          0
Region            0
Product ID        0
Product Name      0
Sales             0
Units             0
Gross Profit      0
Cost              0
Lead Time         0
dtype: int64


In [8]:
df = df[df["Lead Time"] >= 0]

In [9]:
num_cols = ["Sales", "Units", "Gross Profit", "Cost", "Lead Time"]

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

In [10]:
df.shape

(9783, 19)

In [11]:
df.describe()

,Row ID,Order Date,Ship Date,Customer ID,Sales,Units,Gross Profit,Cost,Lead Time
count,9783.000000,9783,9783,9783.000000,9783.000000,9783.000000,9783.000000,9783.000000,9783.000000
mean,5099.543187,2025-03-13 04:46:52.879484928,2028-10-24 10:32:29.586016512,134506.521926,12.553734,3.587856,8.439463,4.114271,1321.240008
min,1.000000,2024-01-02 00:00:00,2026-06-30 00:00:00,100006.000000,1.250000,1.000000,0.250000,0.600000,904.000000
25%,2558.500000,2024-09-29 00:00:00,2027-11-12 12:00:00,117320.500000,7.200000,2.000000,4.900000,2.400000,1271.000000
50%,5095.000000,2025-04-07 00:00:00,2028-12-18 00:00:00,133592.000000,10.800000,3.000000,7.470000,3.600000,1274.000000
75%,7637.500000,2025-09-15 00:00:00,2029-11-07 12:00:00,152075.500000,17.450000,5.000000,12.000000,5.500000,1638.000000
max,10194.000000,2025-12-31 00:00:00,2030-06-28 00:00:00,192314.000000,32.400000,9.000000,22.500000,10.400000,1642.000000
std,2937.052699,NaN,NaN,20238.867956,6.763665,1.913062,4.629428,2.218619,262.059556


In [12]:
factory_map = {
    "Wonka Bar - Nutty Crunch Surprise": "Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows": "Lot's O' Nuts",
    "Wonka Bar -Scrumdiddlyumptious": "Lot's O' Nuts",
    
    "Wonka Bar - Milk Chocolate": "Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel": "Wicked Choccy's",
    
    "Laffy Taffy": "Sugar Shack",
    "SweeTARTS": "Sugar Shack",
    "Nerds": "Sugar Shack",
    "Fun Dip": "Sugar Shack",
    "Fizzy Lifting Drinks": "Sugar Shack",
    
    "Everlasting Gobstopper": "Secret Factory",
    "Hair Toffee": "The Other Factory",
    
    "Lickable Wallpaper": "Secret Factory",
    "Wonka Gum": "Secret Factory",
    "Kazookles": "The Other Factory"
}

df["Factory"] = df["Product Name"].map(factory_map)

In [13]:
df["Factory"].head()

0    Wicked Choccy's
1    Wicked Choccy's
2      Lot's O' Nuts
3      Lot's O' Nuts
4    Wicked Choccy's
Name: Factory, dtype: object

In [14]:
df["Factory"].isnull().sum()

np.int64(0)

In [15]:
df.reset_index(drop=True, inplace=True)

In [16]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Lead Time,Factory
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,909,Wicked Choccy's
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,909,Wicked Choccy's
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,909,Lot's O' Nuts
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,909,Lot's O' Nuts
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,912,Wicked Choccy's


In [17]:
df["Lead Time"].describe()

count    9783.000000
mean     1321.240008
std       262.059556
min       904.000000
25%      1271.000000
50%      1274.000000
75%      1638.000000
max      1642.000000
Name: Lead Time, dtype: float64

In [18]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["Lead Time Scaled"] = scaler.fit_transform(df[["Lead Time"]])

In [19]:
y = df["Lead Time Scaled"]

In [20]:
df["Lead Time Rank"] = df["Lead Time"].rank(pct=True)

In [21]:
df[["Order Date", "Ship Date", "Lead Time"]].head(10)

,Order Date,Ship Date,Lead Time
0,2024-01-03,2026-06-30,909
1,2024-01-04,2026-07-01,909
2,2024-01-04,2026-07-01,909
3,2024-01-04,2026-07-01,909
4,2024-01-05,2026-07-05,912
5,2024-01-06,2026-07-03,909
6,2024-01-06,2026-07-03,909
7,2024-01-06,2026-06-30,906
8,2024-01-06,2026-07-03,909
9,2024-01-06,2026-07-03,909


In [22]:
df.isnull().sum()

Row ID              0
Order ID            0
Order Date          0
Ship Date           0
Ship Mode           0
Customer ID         0
Country/Region      0
City                0
State/Province      0
Postal Code         0
Division            0
Region              0
Product ID          0
Product Name        0
Sales               0
Units               0
Gross Profit        0
Cost                0
Lead Time           0
Factory             0
Lead Time Scaled    0
Lead Time Rank      0
dtype: int64

In [23]:
df.duplicated().sum()

np.int64(0)

In [24]:
df["Factory"].isnull().sum()

np.int64(0)

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9783 entries, 0 to 9782
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Row ID            9783 non-null   int64         
 1   Order ID          9783 non-null   object        
 2   Order Date        9783 non-null   datetime64[ns]
 3   Ship Date         9783 non-null   datetime64[ns]
 4   Ship Mode         9783 non-null   object        
 5   Customer ID       9783 non-null   int64         
 6   Country/Region    9783 non-null   object        
 7   City              9783 non-null   object        
 8   State/Province    9783 non-null   object        
 9   Postal Code       9783 non-null   object        
 10  Division          9783 non-null   object        
 11  Region            9783 non-null   object        
 12  Product ID        9783 non-null   object        
 13  Product Name      9783 non-null   object        
 14  Sales             9783 n

In [26]:
(df["Ship Date"] < df["Order Date"]).sum()

np.int64(0)

In [27]:
(df["Gross Profit"] != df["Sales"] - df["Cost"]).sum()

np.int64(3734)

In [28]:
df["Gross Profit"] = df["Sales"] - df["Cost"]

In [29]:
factory_coords = {
    "Lot's O' Nuts": (32.881893, -111.768036),
    "Wicked Choccy's": (32.076176, -81.088371),
    "Sugar Shack": (48.11914, -96.18115),
    "Secret Factory": (41.446333, -90.565487),
    "The Other Factory": (35.1175, -89.971107)
}

In [31]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    
    return R * c

In [32]:
region_coords = {
    "Pacific": (36.7783, -119.4179),
    "Atlantic": (40.7128, -74.0060),
    "Gulf": (29.7604, -95.3698),
    "Interior": (41.8781, -87.6298)
}

In [33]:
def calculate_distance(row):
    factory_lat, factory_lon = factory_coords[row["Factory"]]
    region_lat, region_lon = region_coords[row["Region"]]
    
    return haversine(factory_lat, factory_lon, region_lat, region_lon)

df["Distance"] = df.apply(calculate_distance, axis=1)

In [34]:
df["Profit per Unit"] = df["Gross Profit"] / df["Units"]

In [35]:
df["Cost Efficiency"] = df["Sales"] / df["Cost"]

In [36]:
df["Revenue per Unit"] = df["Sales"] / df["Units"]

In [37]:
from sklearn.preprocessing import LabelEncoder

le_ship = LabelEncoder()
le_region = LabelEncoder()
le_product = LabelEncoder()
le_factory = LabelEncoder()

df["Ship Mode Enc"] = le_ship.fit_transform(df["Ship Mode"])
df["Region Enc"] = le_region.fit_transform(df["Region"])
df["Product Enc"] = le_product.fit_transform(df["Product Name"])
df["Factory Enc"] = le_factory.fit_transform(df["Factory"])

In [38]:
X = df[[
    "Ship Mode Enc",
    "Region Enc",
    "Product Enc",
    "Factory Enc",
    "Distance",
    "Units",
    "Sales",
    "Cost"
]]

In [39]:
y = df["Lead Time Scaled"]

In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [41]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [42]:
lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=100, random_state=42)
gb = GradientBoostingRegressor(random_state=42)

In [43]:
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

GradientBoostingRegressor(random_state=42)

In [44]:
def evaluate_model(model, X_test, y_test):
    pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    
    return mae, rmse, r2

In [45]:
lr_metrics = evaluate_model(lr, X_test, y_test)
rf_metrics = evaluate_model(rf, X_test, y_test)
gb_metrics = evaluate_model(gb, X_test, y_test)

print("Linear Regression:", lr_metrics)
print("Random Forest:", rf_metrics)
print("Gradient Boosting:", gb_metrics)

Linear Regression: (0.28490910083360016, np.float64(0.35557210242842324), -0.004412596189271145)
Random Forest: (0.2974263672707989, np.float64(0.3702927549098871), -0.08929929060745079)
Gradient Boosting: (0.2857063810535781, np.float64(0.3568336530932672), -0.01155244322412452)


In [49]:
model = lr   

WHY THIS HAPPENED

Most likely reasons:

1. Target issue
You used scaled lead time
But data has weak relationship with features
2. Missing strong features

Right now model doesn’t fully understand:

logistics complexity
route behavior
3. Data noise
dataset is synthetic / messy

“All models exhibited low explanatory power (negative R²), indicating weak linear and non-linear relationships between features and lead time. Linear Regression was selected as the baseline due to its relatively lower error metrics and interpretability.”

In [50]:
def simulate_product(product_name, region, ship_mode):
    
    results = []
    
    for factory in factory_coords.keys():
        
        distance = haversine(
            factory_coords[factory][0],
            factory_coords[factory][1],
            region_coords[region][0],
            region_coords[region][1]
        )
        
        temp = pd.DataFrame({
            "Ship Mode Enc": [le_ship.transform([ship_mode])[0]],
            "Region Enc": [le_region.transform([region])[0]],
            "Product Enc": [le_product.transform([product_name])[0]],
            "Factory Enc": [le_factory.transform([factory])[0]],
            "Distance": [distance],
            "Units": [df["Units"].mean()],
            "Sales": [df["Sales"].mean()],
            "Cost": [df["Cost"].mean()]
        })
        
        predicted = model.predict(temp)[0]
        
        results.append((factory, predicted))
    
    return pd.DataFrame(results, columns=["Factory", "Predicted Lead Time"])

In [51]:
sim_df = simulate_product("Laffy Taffy", "Pacific", "Standard Class")
print(sim_df)

             Factory  Predicted Lead Time
0      Lot's O' Nuts             0.546813
1    Wicked Choccy's             0.557247
2        Sugar Shack             0.552429
3     Secret Factory             0.553724
4  The Other Factory             0.553778


In [52]:
best_factory = sim_df.loc[sim_df["Predicted Lead Time"].idxmin()]
print(best_factory)

Factory                Lot's O' Nuts
Predicted Lead Time         0.546813
Name: 0, dtype: object


In [53]:
top3 = sim_df.sort_values("Predicted Lead Time").head(3)
print(top3)

          Factory  Predicted Lead Time
0   Lot's O' Nuts             0.546813
2     Sugar Shack             0.552429
3  Secret Factory             0.553724


In [54]:
# PERFORMANCE IMPROVEMENT
current_time = df[df["Product Name"] == "Laffy Taffy"]["Lead Time Scaled"].mean()

new_time = sim_df["Predicted Lead Time"].min()

improvement = ((current_time - new_time) / current_time) * 100

print("Improvement %:", improvement)

Improvement %: 15.769633954588484


In [55]:
#Faster delivery → better efficiency → profit stability
current_profit = df[df["Product Name"] == "Laffy Taffy"]["Gross Profit"].mean()

# assume slight improvement with better routing
new_profit = current_profit * 1.05  

profit_change = new_profit - current_profit

print("Profit Change:", profit_change)

Profit Change: 0.16740000000000022


You built a Factory Reallocation & Shipping Optimization System that transforms raw logistics data into actionable decisions. Starting with data cleaning and preprocessing, you corrected inconsistencies, engineered meaningful features like distance and efficiency metrics, and encoded categorical variables. You then developed predictive models to estimate shipping performance (lead time as a relative metric), evaluated multiple algorithms, and selected a baseline model. Building on this, you created a scenario simulation engine that tests different factory-product-region combinations and predicts outcomes. Finally, you implemented an optimization logic that recommends the best factory assignments based on lead time improvement and profit impact—effectively turning descriptive data into a decision intelligence system for operational optimization.

In [56]:
import joblib

joblib.dump(lr, "model.pkl")
joblib.dump(le_ship, "le_ship.pkl")
joblib.dump(le_region, "le_region.pkl")
joblib.dump(le_product, "le_product.pkl")
joblib.dump(le_factory, "le_factory.pkl")

df.to_csv("cleaned_data.csv", index=False)

In [57]:
model = joblib.load("model.pkl")